# 🎓 VKR: Deepfake Detection — GPU Training

**Dual-path spatial-temporal architecture for video-level deepfake detection.**

Этот notebook запускает все 4 эксперимента (A1-A4) на GPU Kaggle.

**Перед запуском:**
1. Загрузите датасет `preprocessed_DFDC02_16` как Kaggle Dataset
2. Загрузите код проекта как второй Dataset (или вставьте файлы)
3. Включите GPU: Settings → Accelerator → GPU P100

## 1. Setup

In [ ]:
!pip install -q timm facenet-pytorch
!nvidia-smi

In [ ]:
import os, sys, shutil, torch

# === НАСТРОЙТЕ ПУТИ ===
# Вариант A: Если код загружен как Kaggle Dataset
CODE_DATASET = '/kaggle/input/deepfake-detection-code'  # Поменяйте на имя вашего dataset
# Вариант B: Если код в /kaggle/working (через git clone)
# CODE_DATASET = '/kaggle/working/deepfake-detection'

# Путь к preprocessed данным
DATA_DATASET = '/kaggle/input/preprocessed-dfdc02-16'  # Поменяйте на имя вашего dataset

# Рабочая директория (writable)
WORK_DIR = '/kaggle/working/project'
os.makedirs(WORK_DIR, exist_ok=True)

# Копируем код в writable директорию
if os.path.exists(CODE_DATASET):
    for item in os.listdir(CODE_DATASET):
        src = os.path.join(CODE_DATASET, item)
        dst = os.path.join(WORK_DIR, item)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
    print(f'Code copied to {WORK_DIR}')
else:
    print(f'WARNING: {CODE_DATASET} not found!')

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

print(f'Working dir: {os.getcwd()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"}')
print(f'CUDA: {torch.cuda.is_available()}')
print(f'Files: {os.listdir(".")}')

In [ ]:
# Проверяем датасет
data_path = DATA_DATASET

# Если датасет внутри подпапки
for candidate in [
    DATA_DATASET,
    os.path.join(DATA_DATASET, 'preprocessed_DFDC02_16'),
    os.path.join(DATA_DATASET, 'data', 'preprocessed_data', 'preprocessed_DFDC02_16'),
]:
    if os.path.exists(os.path.join(candidate, 'real')) or os.path.exists(os.path.join(candidate, 'fake')):
        data_path = candidate
        break

real_count = len(os.listdir(os.path.join(data_path, 'real'))) if os.path.exists(os.path.join(data_path, 'real')) else 0
fake_count = len(os.listdir(os.path.join(data_path, 'fake'))) if os.path.exists(os.path.join(data_path, 'fake')) else 0

print(f'Dataset path: {data_path}')
print(f'Real videos: {real_count}')
print(f'Fake videos: {fake_count}')
print(f'Total: {real_count + fake_count}')

## 2. Config

In [ ]:
# Гиперпараметры
MAX_EPOCHS = 30        # Больше эпох на GPU — можно себе позволить
BATCH_SIZE = 16        # На P100 16GB можно увеличить
PATIENCE = 7           # Больше терпения с быстрым GPU
NUM_WORKERS = 2        # Ускорение загрузки данных

print(f'Epochs: {MAX_EPOCHS}, Batch: {BATCH_SIZE}, Patience: {PATIENCE}')

## 3. Run All Experiments

In [ ]:
from config import Config
from train import train
from utils import save_metrics
import time, json

EXPERIMENTS = [
    {'name': 'A1_full_model', 'model_type': 'full', 'fusion_type': 'adaptive',
     'desc': 'Полная модель (dual-path + adaptive fusion)'},
    {'name': 'A2_spatial_only', 'model_type': 'spatial', 'fusion_type': 'adaptive',
     'desc': 'Spatial-only (ablation)'},
    {'name': 'A3_temporal_only', 'model_type': 'temporal', 'fusion_type': 'adaptive',
     'desc': 'Temporal-only (ablation)'},
    {'name': 'A4_sequential', 'model_type': 'sequential', 'fusion_type': 'adaptive',
     'desc': 'Sequential CNN→BiLSTM (ablation)'},
]

all_results = []

for i, exp in enumerate(EXPERIMENTS, 1):
    print(f"\n{'='*70}")
    print(f"[{i}/4] {exp['name']}: {exp['desc']}")
    print(f"{'='*70}\n")
    
    cfg = Config()
    cfg.dataset_root = data_path
    cfg.model_type = exp['model_type']
    cfg.fusion_type = exp['fusion_type']
    cfg.max_epochs = MAX_EPOCHS
    cfg.batch_size = BATCH_SIZE
    cfg.patience = PATIENCE
    cfg.device = 'auto'  # Will pick CUDA
    cfg.use_amp = True   # Mixed precision on GPU!
    cfg.num_workers = NUM_WORKERS
    cfg.pin_memory = True
    cfg.output_dir = './experiments'
    
    t0 = time.time()
    try:
        metrics = train(cfg)
        metrics['experiment_name'] = exp['name']
        metrics['description'] = exp['desc']
        metrics['status'] = 'success'
        metrics['duration_min'] = round((time.time() - t0) / 60, 1)
        all_results.append(metrics)
        
        test = metrics.get('test', {})
        print(f"\n✅ {exp['name']} done in {metrics['duration_min']} min")
        print(f"   Test AUC={test.get('auc',0):.4f} Acc={test.get('accuracy',0):.4f} F1={test.get('f1',0):.4f}")
    except Exception as e:
        print(f"\n❌ {exp['name']} FAILED: {e}")
        all_results.append({'experiment_name': exp['name'], 'status': 'failed', 'error': str(e)})
    
    # Сохраняем после каждого эксперимента (на случай crash)
    save_metrics(all_results, './experiments/all_results.json')
    print(f"Results saved to ./experiments/all_results.json")

## 4. Results Summary

In [ ]:
print(f"\n{'='*70}")
print("ИТОГОВАЯ СВОДКА")
print(f"{'='*70}")
print(f"{'Experiment':<25} {'AUC':>8} {'Acc':>8} {'F1':>8} {'EER':>8} {'Time':>8}")
print('-' * 70)

for r in all_results:
    if r.get('status') == 'success':
        t = r.get('test', {})
        print(f"{r['experiment_name']:<25} {t.get('auc',0):>8.4f} {t.get('accuracy',0):>8.4f} "
              f"{t.get('f1',0):>8.4f} {t.get('eer',0):>8.4f} {r.get('duration_min',0):>7.1f}m")
    else:
        print(f"{r['experiment_name']:<25} {'FAILED':>8}")

print(f"{'='*70}")

## 5. Generate Plots

In [ ]:
from plot_training_curves import load_experiment_metrics, plot_loss_curves, plot_auc_curves, plot_test_comparison
import matplotlib.pyplot as plt

os.makedirs('./experiments/plots', exist_ok=True)
results = load_experiment_metrics('./experiments')
print(f'Found {len(results)} experiments')

if results:
    plot_loss_curves(results, './experiments/plots')
    plot_auc_curves(results, './experiments/plots')
    plot_test_comparison(results, './experiments/plots')
    print('All plots saved!')

In [ ]:
# Показать графики
from IPython.display import Image, display

for plot_name in ['loss_curves.png', 'val_auc_curves.png', 'test_comparison.png']:
    path = f'./experiments/plots/{plot_name}'
    if os.path.exists(path):
        print(f'\n--- {plot_name} ---')
        display(Image(filename=path))

In [ ]:
# Показать confusion matrix и ROC для каждого эксперимента
from IPython.display import Image, display
import glob

for img_path in sorted(glob.glob('./experiments/dfdc02_*/confusion_matrix_test.png')):
    print(f'\n--- {os.path.basename(os.path.dirname(img_path))} ---')
    display(Image(filename=img_path))

for img_path in sorted(glob.glob('./experiments/dfdc02_*/roc_curve_test.png')):
    print(f'\n--- {os.path.basename(os.path.dirname(img_path))} ---')
    display(Image(filename=img_path))

## 6. Download Results

После завершения скачайте:
- `experiments/all_results.json` — все метрики
- `experiments/*/best_model.pt` — чекпоинты моделей
- `experiments/*/metrics.json` — метрики по каждой модели
- `experiments/*/confusion_matrix_test.png` — confusion matrix
- `experiments/*/roc_curve_test.png` — ROC кривые
- `experiments/plots/` — сводные графики

In [ ]:
# Создаём архив для скачивания
import shutil

# Архив всех результатов (без тяжёлых чекпоинтов)
!cd /kaggle/working/project && tar czf /kaggle/working/results_light.tar.gz \
    experiments/all_results.json \
    experiments/results_table.txt \
    experiments/plots/ \
    experiments/dfdc02_*/metrics.json \
    experiments/dfdc02_*/confusion_matrix_test.png \
    experiments/dfdc02_*/roc_curve_test.png \
    experiments/dfdc02_*/predictions.csv \
    2>/dev/null

# Архив с чекпоинтами (для инференса)
!cd /kaggle/working/project && tar czf /kaggle/working/checkpoints.tar.gz \
    experiments/dfdc02_*/best_model.pt \
    2>/dev/null

print('Archives created:')
!ls -lh /kaggle/working/results_light.tar.gz /kaggle/working/checkpoints.tar.gz